# 07 · 對齊 ①：SFT 監督式微調

到上一課為止，我們的 GPT 只會「接字」——你給開頭，它接著胡謅。但 ChatGPT 會**照你的指令回答**。從「會接字」到「會照指令回答」的第一步，就是 **SFT（Supervised Fine-Tuning，監督式微調）**：拿一堆「指令 → 理想回應」的配對，繼續訓練模型。

## 學習目標

- 理解 base 模型（只會接字）與對齊後模型（會照指令回答）的差別
- 準備「指令 → 回應」資料集
- 對 base 模型做 SFT，看它學會照格式回答

## 1. 準備資料

我們用一個小到能驗證的任務：**單位數加法**，固定模板 `問：a加b等於 答：c。`。這 100 個配對就是我們的「指令資料集」。

In [1]:
import torch
torch.manual_seed(0)
poems = """床前明月光，疑是地上霜。舉頭望明月，低頭思故鄉。
春眠不覺曉，處處聞啼鳥。夜來風雨聲，花落知多少。
白日依山盡，黃河入海流。欲窮千里目，更上一層樓。
紅豆生南國，春來發幾枝。願君多采擷，此物最相思。
空山不見人，但聞人語響。返景入深林，復照青苔上。
千山鳥飛絕，萬徑人蹤滅。孤舟蓑笠翁，獨釣寒江雪。
松下問童子，言師采藥去。只在此山中，雲深不知處。
人閒桂花落，夜靜春山空。月出驚山鳥，時鳴春澗中。
君自故鄉來，應知故鄉事。來日綺窗前，寒梅著花未。
獨坐幽篁裡，彈琴復長嘯。深林人不知，明月來相照。
山中相送罷，日暮掩柴扉。春草明年綠，王孫歸不歸。
功蓋三分國，名成八陣圖。江流石不轉，遺恨失吞吳。
前不見古人，後不見來者。念天地之悠悠，獨愴然而涕下。
葡萄美酒夜光杯，欲飲琵琶馬上催。醉臥沙場君莫笑，古來征戰幾人回。
秦時明月漢時關，萬里長征人未還。但使龍城飛將在，不教胡馬度陰山。
朝辭白帝彩雲間，千里江陵一日還。兩岸猿聲啼不住，輕舟已過萬重山。
故人西辭黃鶴樓，煙花三月下揚州。孤帆遠影碧空盡，唯見長江天際流。
月落烏啼霜滿天，江楓漁火對愁眠。姑蘇城外寒山寺，夜半鐘聲到客船。
獨在異鄉為異客，每逢佳節倍思親。遙知兄弟登高處，遍插茱萸少一人。
日照香爐生紫煙，遙看瀑布掛前川。飛流直下三千尺，疑是銀河落九天。
國破山河在，城春草木深。感時花濺淚，恨別鳥驚心。
岐王宅裡尋常見，崔九堂前幾度聞。正是江南好風景，落花時節又逢君。
渭城朝雨浥輕塵，客舍青青柳色新。勸君更盡一杯酒，西出陽關無故人。
清明時節雨紛紛，路上行人欲斷魂。借問酒家何處有，牧童遙指杏花村。
千里黃雲白日曛，北風吹雁雪紛紛。莫愁前路無知己，天下誰人不識君。
"""

# 指令資料集：單位數加法，固定模板（這就是「指令→回應」配對）
pairs = [(a, b) for a in range(10) for b in range(10)]
instr = "".join(f"問：{a}加{b}等於 答：{a + b}。\n" for a, b in pairs)

# 詞表涵蓋「詩 + 指令」的所有字元
text = poems + instr
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
poem_data = torch.tensor(encode(poems), dtype=torch.long)
instr_data = torch.tensor(encode(instr), dtype=torch.long)
print(f"詞表 {vocab_size}　詩語料 {len(poem_data)} 字　指令語料 {len(instr_data)} 字")

詞表 344　詩語料 715 字　指令語料 1345 字


In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class MultiHeadAttention(nn.Module):
    """因果多頭自注意力（一次算完所有 head）。"""
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)   # 一次產生 Q,K,V
        self.proj = nn.Linear(n_embd, n_embd)
        self.drop = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * self.head_dim ** -0.5
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))  # 因果遮罩
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.proj(y))

class Block(nn.Module):
    """Transformer block：注意力 + 前饋，各帶殘差與 LayerNorm。"""
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # 殘差連接
        x = x + self.ff(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=3, block_size=48, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)   # token 嵌入
        self.pos_emb = nn.Embedding(block_size, n_embd)   # 位置嵌入
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)         # 預測下一個 token

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.ln_f(self.blocks(x))
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]      # 只看最近 block_size 個 token
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature   # 取最後一步
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)         # 依機率抽樣
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [3]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
block_size = 48

def train_on(model, data, steps, lr=3e-4):
    data = data.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(steps):
        ix = torch.randint(len(data) - block_size - 1, (32,))
        x = torch.stack([data[i:i + block_size] for i in ix])
        y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
        _, loss = model(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

def answer(model, q):                      # 餵入「問：a加b等於 答：」，讓它續寫答案
    idx = torch.tensor([encode(q)], device=device)
    out = model.generate(idx, max_new_tokens=6, temperature=0.1)[0].tolist()
    return decode(out)

## 2. Base 模型：只會接字，不會回答

先在「詩」語料上訓練一個 base 模型。它學會寫詩的味道，但你問它算術，它只會continue胡謅。

In [4]:
base = MiniGPT(vocab_size, n_embd=128, n_head=4, n_layer=3, block_size=block_size).to(device)
train_on(base, poem_data, steps=1500)
print("Base 模型回答「問：3加4等於 答：」：")
print("  ", answer(base, "問：3加4等於 答："))
print("（base 沒看過指令格式，只會接字、答非所問）")

Base 模型回答「問：3加4等於 答：」：


   問：3加4等於 答：盡，唯見長江
（base 沒看過指令格式，只會接字、答非所問）


## 3. SFT：在指令資料上繼續訓練

拿同一個模型，**繼續**在「指令 → 回應」配對上訓練。這一步就是 SFT。

In [5]:
import copy
sft = copy.deepcopy(base)               # 從 base 出發
train_on(sft, instr_data, steps=2500)   # 在指令資料上微調

print("SFT 後，同樣問題：")
for q in ["問：3加4等於 答：", "問：5加2等於 答：", "問：8加6等於 答："]:
    print("  ", answer(sft, q))

SFT 後，同樣問題：
   問：3加4等於 答：7。
問：3
   問：5加2等於 答：8。
問：5
   問：8加6等於 答：14。
問：


微調後，模型**學會了照格式回答**，而且小資料的加法多半答得對（它其實把這 100 題背了起來）。功能很陽春沒關係——重點是你看見了**對齊的第一步**：用「指令→回應」資料，把一個只會接字的 base 模型，調教成會聽指令做事的模型。真實的 ChatGPT 就是用同樣的 SFT（加上海量、多樣的指令資料）煉成的。

## 小結

- **Base 模型**只會接字（語言建模）；**SFT** 教它照「指令→回應」格式回答。
- SFT = 拿配對資料、用一樣的訓練機制繼續微調。
- 這是「對齊（alignment）」的第一步——讓模型做**人類想要**的事。

## 練習

1. 把指令任務換成減法或「你好→你好！很高興見到你」之類的問候，SFT 後它學得會嗎？
2. SFT 步數調太多會怎樣？（提示：它會更死記、對沒見過的題型更不會舉一反三）

最後一課，用 **DPO** 做對齊的第二步：讓模型對齊**人類偏好**。